# Agentic RAG Notebook Showcase (Standalone Deliverable)

This notebook demonstrates a standalone LangGraph agentic RAG runtime implemented entirely inside `demo_notebook/runtime`.

Goals:
- Show orchestrator routing (BASIC vs AGENT).
- Show multi-agent graph orchestration and tool calling.
- Show explicit GeneralAgent path without memory tools.
- Use print-based observability (no external observability stack required).

## 0) One-time setup

Run in terminal from `demo_notebook/`:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -U pip
python -m pip install -r requirements.txt
cp .env.example .env
python scripts/check_isolation.py
```

In [1]:
!python -m pip install -r requirements.txt

In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv

from runtime.config import load_settings
from runtime.providers import build_provider_bundle
from runtime.stores import PostgresVectorStore
from runtime.observability import PrintTraceCallbackHandler
from runtime.orchestrator import DemoOrchestrator

env_path = Path('.env')
if env_path.exists():
    load_dotenv(env_path)

settings = load_settings(str(env_path) if env_path.exists() else None)
print(f'Provider mode: {settings.provider_mode}')
print(f'KB dir: {settings.kb_dir.resolve()}')
print(f'Embedding dim: {settings.embedding_dim}')

Provider mode: ollama
KB dir: /Users/shivbalodi/Desktop/Rag_Research/langchain_agentic_chatbot_v2/data/kb
Embedding dim: 768


In [ ]:
import psycopg2

print("Using DSN:", settings.pg_dsn)

with psycopg2.connect(settings.pg_dsn) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS dn_chunks CASCADE;")
        cur.execute("DROP TABLE IF EXISTS dn_documents CASCADE;")
        cur.execute("SELECT to_regclass('public.dn_chunks'), to_regclass('public.dn_documents');")
        print("Post-drop:", cur.fetchone())

In [2]:
providers = build_provider_bundle(settings)
store = PostgresVectorStore(settings, providers.embeddings)
trace = PrintTraceCallbackHandler(enabled=True, prefix='NOTEBOOK')
orchestrator = DemoOrchestrator(settings, providers, store, callback_handler=trace)

bootstrap = orchestrator.bootstrap_kb(reindex=False)
print('KB bootstrap:', bootstrap)
docs = store.list_documents()
print(f'Indexed docs: {len(docs)}')

RuntimeError: dn_chunks.embedding uses vector(1536) but NOTEBOOK_EMBEDDING_DIM=768. Drop dn_chunks and dn_documents to reinitialize notebook schema.

## A) BASIC route demo

In [ ]:
from runtime.scenarios import BASIC_SCENARIO

result = orchestrator.process_turn(BASIC_SCENARIO, force_agent=False, stream_updates=False)
print('Route:', result.route)
print('Used fallback:', result.used_fallback)
print('--- Answer ---')
print(result.answer)

## B) AGENT RAG citation demo

In [ ]:
from runtime.scenarios import RAG_CITATION_SCENARIO

result = orchestrator.process_turn(RAG_CITATION_SCENARIO, force_agent=True, stream_updates=False)
print('Route:', result.route)
print('Used fallback:', result.used_fallback)
print('--- Answer ---')
print(result.answer)

## C) Parallel multi-doc orchestration trace

This run prints graph node updates (supervisor decisions, planner/worker/synthesizer activity).

In [ ]:
from runtime.scenarios import PARALLEL_COMPARE_SCENARIO

result = orchestrator.process_turn(PARALLEL_COMPARE_SCENARIO, force_agent=True, stream_updates=True)
print('Route:', result.route)
print('Used fallback:', result.used_fallback)
print('--- Answer ---')
print(result.answer)

## D) Explicit GeneralAgent path (no memory tools)

In [ ]:
from runtime.scenarios import GENERAL_AGENT_SCENARIO

answer = orchestrator.run_general_agent_direct(GENERAL_AGENT_SCENARIO)
print('--- GeneralAgent answer ---')
print(answer)

## E) Provider switching notes

Set `NOTEBOOK_PROVIDER` in `.env` and restart kernel:

- `azure`: uses Azure chat/judge/embeddings
- `ollama`: uses Ollama chat/judge/embeddings
- `vllm`: uses OpenAI-compatible chat/judge, plus optional OpenAI-compatible embeddings or local fallback

For vLLM without embeddings endpoint, keep:

```env
NOTEBOOK_VLLM_USE_OPENAI_EMBEDDINGS=false
```

## F) Skills Showcase (baseline vs skills-on)

This section runs the same scenario twice to show prompt-control deltas.
The runtime remains isolated; only `demo_notebook/skills/*.md` is applied in showcase mode.


In [ ]:
from dataclasses import replace
from runtime.scenarios import SKILLS_SHOWCASE_SCENARIO

print('Scenario:')
print(SKILLS_SHOWCASE_SCENARIO)

baseline_settings = replace(settings, skills_enabled=False, skills_showcase_mode=False)
baseline_orchestrator = DemoOrchestrator(baseline_settings, providers, store, callback_handler=trace)
baseline_result = baseline_orchestrator.process_turn(SKILLS_SHOWCASE_SCENARIO, force_agent=True, stream_updates=False)

print('\n=== Baseline (skills disabled) ===')
print('Route:', baseline_result.route)
print('Used fallback:', baseline_result.used_fallback)
print(baseline_result.answer)

skills_settings = replace(settings, skills_enabled=True, skills_showcase_mode=True)
skills_orchestrator = DemoOrchestrator(skills_settings, providers, store, callback_handler=trace)
skills_result = skills_orchestrator.process_turn(SKILLS_SHOWCASE_SCENARIO, force_agent=True, stream_updates=False)

print('\n=== Skills mode (showcase enabled) ===')
print('Route:', skills_result.route)
print('Used fallback:', skills_result.used_fallback)
print('Active skill files:')
if skills_orchestrator.active_skill_files:
    for skill_path in skills_orchestrator.active_skill_files:
        print('-', skill_path)
else:
    print('- none found')
print(skills_result.answer)
